In [13]:
import torch
from torch.utils.data import DataLoader, TensorDataset
import json

with open('move_to_id.json', 'r') as f:
    move_to_id = json.load(f)
PAD_ID = move_to_id["<PAD>"]

# Load the single tensor
encoded_tensor = torch.load('encoded_games_small.pt')

# Shift for X and Y
X_test = encoded_tensor[:, :-1]
Y_test = encoded_tensor[:, 1:]

test_dataset = TensorDataset(X_test, Y_test)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)


In [14]:
import torch.nn as nn

class SparseMultiheadAttention(nn.Module):
    def __init__(self, d_model, nhead, sparsity_window=32):
        super().__init__()
        self.nhead = nhead
        self.sparsity_window = sparsity_window
        self.attn = nn.MultiheadAttention(d_model, nhead, batch_first=True)

    def forward(self, x):
        T = x.size(1)

        # Sparse mask: prevent attending to tokens beyond a sliding window

        mask = torch.ones((T, T), device=x.device, dtype=torch.bool)
        mask = torch.triu(torch.ones(T,T,device=x.device, dtype=torch.bool), diagonal=1)

        if self.sparsity_window < T:
            for i in range(T):
                start = max(0, i - self.sparsity_window)
                mask[i, :start] = True

        out, _ = self.attn(x, x, x, attn_mask=mask)
        out = torch.nan_to_num(out)
        return out

class SparseDecoderLayer(nn.Module):
    def __init__(self, d_model, nhead, dim_feedforward=2048, sparsity_window=32, dropout=0.1):
        super().__init__()
        self.self_attn = SparseMultiheadAttention(d_model, nhead, sparsity_window)
        self.linear1 = nn.Linear(d_model, dim_feedforward)
        self.linear2 = nn.Linear(dim_feedforward, d_model)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        attn_out = self.self_attn(x)
        x = x + self.dropout(attn_out)
        x = self.norm1(x)

        ff_out = self.linear2(F.relu(self.linear1(x)))
        x = x + self.dropout(ff_out)
        x = self.norm2(x)

        return x

class ExpandedAttentionChessDecoder(nn.Module):
    def __init__(self, vocab_size, d_model, nhead, num_layers, max_len, sparsity_window=32):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, d_model)
        self.pos_emb = nn.Embedding(max_len, d_model)

        self.layers = nn.ModuleList([
            SparseDecoderLayer(d_model, nhead, sparsity_window=sparsity_window)
            for _ in range(num_layers)
        ])

        self.fc_out = nn.Linear(d_model, vocab_size)

    def forward(self, x):
        B, T = x.size()
        positions = torch.arange(0, T, device=x.device).unsqueeze(0)
        x = self.embed(x) + self.pos_emb(positions)

        for layer in self.layers:
            x = layer(x)

        return self.fc_out(x)

In [15]:
vocab_size = len(move_to_id)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'On device {device}')

with open("best_hparams_ea.json", "r") as f:
    best_hparams = json.load(f)

params = best_hparams["best_params"]

# Rebuild model (same as in training)
model = ExpandedAttentionChessDecoder(
    vocab_size=vocab_size,
    d_model=params["d_model"],  # Commented out for testing
    nhead=params["nhead"],
    num_layers=params["num_layers"],  # Commented out for testing
    max_len=encoded_tensor.size(1),
    sparsity_window=32  # can adjust to 32 for even higher sparsity
)

# Load weights if you saved them
model.load_state_dict(torch.load("quant_8bit_epoch5.pt", map_location=device))

model.eval()


On device cuda


ExpandedAttentionChessDecoder(
  (embed): Embedding(11017, 512)
  (pos_emb): Embedding(200, 512)
  (layers): ModuleList(
    (0-5): 6 x SparseDecoderLayer(
      (self_attn): SparseMultiheadAttention(
        (attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=512, out_features=512, bias=True)
        )
      )
      (linear1): Linear(in_features=512, out_features=2048, bias=True)
      (linear2): Linear(in_features=2048, out_features=512, bias=True)
      (norm1): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
      (norm2): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
  )
  (fc_out): Linear(in_features=512, out_features=11017, bias=True)
)

In [16]:
import torch.nn.functional as F

all_preds = []
all_labels = []
total_loss = 0
total_tokens = 0
model.to(device)

with torch.no_grad():
    for x, y in test_loader:
        x = x.to(device)
        y = y.to(device)

        logits = model(x)

        loss = F.cross_entropy(
            logits.reshape(-1, vocab_size),
            y.reshape(-1),
            ignore_index=PAD_ID,
            reduction='sum'
        )
        total_loss += loss.item()
        total_tokens += (y != PAD_ID).sum().item()

        preds = torch.argmax(logits, dim=-1)

        # mask out padding so it doesn't affect metrics
        mask = (y != PAD_ID).reshape(-1)
        all_preds.extend(preds.reshape(-1)[mask].cpu().numpy())
        all_labels.extend(y.reshape(-1)[mask].cpu().numpy())

avg_loss = total_loss / total_tokens

In [17]:
from sklearn.metrics import accuracy_score, f1_score, cohen_kappa_score

acc = accuracy_score(all_labels, all_preds)
f1 = f1_score(all_labels, all_preds, average='macro')  # or 'weighted'
kappa = cohen_kappa_score(all_labels, all_preds)

print(f"Test Loss: {avg_loss:.4f}")
print(f"Accuracy: {acc:.4f}")
print(f"F1 Score: {f1:.4f}")
print(f"Cohen's Kappa: {kappa:.4f}")


Test Loss: 529.7234
Accuracy: 0.0147
F1 Score: 0.0000
Cohen's Kappa: 0.0000
